# MOSAIC: Global Optimization Phase (clustering, lexicographic $\alpha \gg \beta$)

Reads `exp_global.py`'s output and plots the clustering-quality metrics
(see `src/global_opt.py`): pairwise F1, precision, recall, ARI and AMI
(all via the shared `plot_metric` below; only AMI is active by default,
the rest commented out), plus a mean-cluster-count diagnostic, for MOSAIC
(W-2/W-3) against two baselines: the local-IDM "no update" ablation and
the naive MLE point-estimate baseline (oracle $k$, see
`cluster_1d_wcss_optimal`).

Facet grid mirrors `plot_local.ipynb` (the local phase): rows = ($E$,
$S$) combinations, columns = ($\rho$, $\alpha$) combinations, $N$ (sample
size) on the x-axis, so this keeps working unchanged if `conf_global.yaml`
is extended to sweep `n_clients` or `prob_shift`/`alpha` too, not just $S$
and $N$ as it currently does.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.insert(0, str(Path().resolve().parents[1]))
from src.config import *  # noqa
from src.utils import *  # noqa
from src.mosaic import *  # noqa


In [ ]:
# Read config file
config = load_config("conf_global.yaml")

# Results folder: a single one for the whole hyperparameter grid (see
# exp_global.py); each row of df_tot.csv already carries its own
# n_clients/ess/prob_shift/alpha combination.
res_path = Path(config["res_path"])

df_tot = pd.read_csv(res_path / "df_tot.csv")
# df_tot.head()


## Aggregation

Every numeric column already lives at the (hyperparameter combination x
repetition) level; group by every grid hyperparameter (not just the ones
that vary in the current `conf_global.yaml`, so this still works if the
grid is extended later) and take mean/std across repetitions.


In [ ]:
group_cols = ["n_clients", "ess", "prob_shift", "alpha", "size"]
metric_cols = [c for c in df_tot.columns if c not in group_cols + ["rep"]]

# Both mean+std and median+Q1/Q3 are computed for every metric, so
# CENTRAL_STAT (see the style-setup cell) can switch between them later
# without re-aggregating; a plot must never pair mean with quartiles or
# median with std.
g = df_tot.groupby(group_cols)[metric_cols]
agg_mean = g.mean()
agg_mean.columns = [f"{c}_mean" for c in agg_mean.columns]
agg_std = g.std()
agg_std.columns = [f"{c}_std" for c in agg_std.columns]
agg_median = g.median()
agg_median.columns = [f"{c}_median" for c in agg_median.columns]
agg_q1 = g.quantile(0.25)
agg_q1.columns = [f"{c}_q1" for c in agg_q1.columns]
agg_q3 = g.quantile(0.75)
agg_q3.columns = [f"{c}_q3" for c in agg_q3.columns]

df_mean = pd.concat([agg_mean, agg_std, agg_median, agg_q1, agg_q3], axis=1).reset_index()
# df_mean.head()


In [ ]:
sns.reset_defaults()
plt.style.use("seaborn-v0_8-paper")

plt.rcParams.update(
    {
        # ---- base ----
        "figure.figsize": [7, 4],
        "font.size": 12,
        # ---- axes ----
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "axes.linewidth": 0.8,
        "axes.edgecolor": "black",
        # ---- legend ----
        "legend.fontsize": 9,
        # "legend.labelspacing": 0.3,
        # ---- ticks ----
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        # ---- lines ----
        "lines.linewidth": 1.0,
        # ---- dpi ----
        "figure.dpi": 300,
        "savefig.dpi": 300,
        # # ---- tex ----
        "text.usetex": True,
    }
)

# Per-subplot size (inches), shared by every hyperparameter-grid plot below;
# the total figure size is this times the grid's row/column count. Same
# values as plot_global.ipynb, kept here as plain constants to tweak by hand.
SUBPLOT_WIDTH = 4
SUBPLOT_HEIGHT = 2.5

# Fontsize of each facet's column title (top row only) and row label (the
# first line of the leftmost column's y-axis label).
COL_TITLE_FONTSIZE = 12
ROW_LABEL_FONTSIZE = 12

# Across-repetition uncertainty is shown as sparse dot-whisker markers on
# the mean/median line, not as a second overlapping fill_between band (a
# continuous band stays reserved for Grafico 1's mean-to-max credal-set
# width, which is a different kind of spread, not a repetition statistic).
# Which across-repetition statistic every plot's central curve and
# whisker spread use together: "mean" pairs with std (symmetric), "median"
# pairs with Q1/Q3 (asymmetric). A curve is always one or the other, never
# mean+quartiles or median+std.
CENTRAL_STAT = "median"  # "mean" or "median"

N_WHISKER_MARKERS = 5  # how many markers per curve
WHISKER_DODGE = 8  # horizontal offset (in N units) between methods' markers

# Global knobs for every plot below: one line width for all curves
# (solid method lines and dashed references alike), one for the whisker
# markers' error bars (including their caps).
CURVE_LINEWIDTH = 1.4
WHISKER_LINEWIDTH = 1


def add_std_whiskers(ax, x, y, err, color, dodge=0.0, n_markers=N_WHISKER_MARKERS, clip_lower=None):
    """
    Sparse dot-whisker markers for the across-repetition spread around a
    mean/median line. `err` is either a symmetric half-width (e.g. std) or
    an (lower_half, upper_half) pair of arrays for an asymmetric interval
    (e.g. an IQR). `clip_lower` floors the lower whisker (e.g. for a
    log-scale axis, where a non-positive bound would be invalid).
    """
    x, y = np.asarray(x), np.asarray(y)
    idx = np.linspace(0, len(x) - 1, n_markers).astype(int)
    wx, wy = x[idx] + dodge, y[idx]
    if isinstance(err, tuple):
        lo_half, hi_half = np.asarray(err[0])[idx], np.asarray(err[1])[idx]
    else:
        lo_half = hi_half = np.asarray(err)[idx]
    lo, hi = wy - lo_half, wy + hi_half
    if clip_lower is not None:
        lo = np.clip(lo, clip_lower, None)
    ax.errorbar(
        wx, wy, yerr=[wy - lo, hi - wy], fmt="o", color=color, markersize=3,
        markeredgecolor="white", markeredgewidth=0.4,
        elinewidth=WHISKER_LINEWIDTH, capsize=2.5, capthick=WHISKER_LINEWIDTH, zorder=5,
    )


def center_spread(sub, col):
    """
    (center, spread) for `col`'s across-repetition aggregate, matched to
    CENTRAL_STAT: mean+std (spread is a single symmetric half-width) or
    median+Q1/Q3 (spread is a (lower_half, upper_half) pair), reading the
    columns produced by the aggregation cell above. `spread` is ready to
    pass straight to add_std_whiskers.
    """
    if CENTRAL_STAT == "mean":
        center = sub[f"{col}_mean"]
        spread = sub[f"{col}_std"]
    elif CENTRAL_STAT == "median":
        center = sub[f"{col}_median"]
        spread = (center - sub[f"{col}_q1"], sub[f"{col}_q3"] - center)
    else:
        raise ValueError(f"Unknown CENTRAL_STAT: {CENTRAL_STAT!r}")
    return center, spread


## Methods and facets

Same facet convention as the local phase's `plot_local.ipynb`: rows =
($E$, $S$) combinations, columns = ($\rho$, $\alpha$) combinations. $N$
(sample size) is the x-axis, since `conf_global.yaml` now sweeps it
explicitly (an exploratory sweep found MOSAIC's gain over the no-update
ablation depends on $S$ *relative to* $N$, not $S$ alone; see
`cap6_extract.tex`/`conf_global.yaml`'s own comments). Column titles show
just $S$ when `n_clients` has a single value (as now), and both $E$ and
$S$ once it is actually swept, to stay unambiguous.


In [ ]:
# Canonical order everywhere (draw order, legend order, and dodge slots):
# MLE/True, then IDM/noupdate, then MOSAIC W-2, then W-3.
METHODS = ("mle", "noupdate", "mos_w2", "mos_w3")
method_colors = {
    "mle": "grey",
    "noupdate": "#fc9167",
    "mos_w2": "#4fc685",
    "mos_w3": "#9a97f4",
}
method_labels = {
    "mle": "MLE (with oracle)",
    "noupdate": "IDM",
    "mos_w2": "MOSAIC (W-2: intersection)",
    "mos_w3": "MOSAIC (W-3: JSD-based)",
}
# Fixed dodge slot per method (as a multiple of WHISKER_DODGE), identical
# across every plot in both notebooks; see plot_local.ipynb's mos_dodge.
method_dodge = {"mle": -1.5, "noupdate": -0.5, "mos_w2": 0.5, "mos_w3": 1.5}

# Facet grid: rows = (n_clients, ess) combinations, columns =
# (prob_shift, alpha) combinations, same convention as plot_local.ipynb.
row_vals = sorted(set(df_mean[["n_clients", "ess"]].itertuples(index=False, name=None)))
col_vals = sorted(
    set(df_mean[["prob_shift", "alpha"]].itertuples(index=False, name=None)),
    key=lambda pa: (pa[0], -pa[1]),
)
print(
    f"Grid: {len(row_vals)} row(s) (n_clients, ess) x {len(col_vals)} col(s) (prob_shift, alpha)"
)

_n_clients_vals = sorted(df_mean["n_clients"].unique())


def row_label(nc, ess):
    # Show E too once n_clients is actually swept (more than one value);
    # otherwise stay as compact as the local phase's own row labels.
    return fr"$E={nc}$, $S={ess}$" if len(_n_clients_vals) > 1 else fr"$S={ess}$"


def col_label(p, a):
    # alpha is irrelevant (and deduplicated away, see exp_local.hyperparameter_combos)
    # whenever prob_shift == 0.0.
    return fr"$\rho={p}$" if p == 0 else fr"$\rho={p}$, $\alpha={a}$"


def plot_metric(metric, ylabel, ylim=None, hline=None):
    fig, axes = plt.subplots(
        len(row_vals),
        len(col_vals),
        figsize=(SUBPLOT_WIDTH * len(col_vals), SUBPLOT_HEIGHT * len(row_vals)),
        squeeze=False,
        sharex=True,
        sharey=True,
    )
    for i, (nc, ess) in enumerate(row_vals):
        for j, (p, a) in enumerate(col_vals):
            ax = axes[i][j]
            sub = df_mean[
                (df_mean["prob_shift"] == p)
                & (df_mean["alpha"] == a)
                & (df_mean["n_clients"] == nc)
                & (df_mean["ess"] == ess)
            ].sort_values("size")

            if hline is not None:
                ax.axhline(hline, color="gray", linewidth=0.7, linestyle=":")
            for m in METHODS:
                mean, spread = center_spread(sub, f"{m}_{metric}")
                if m == "mle":
                    linestyle = "--"
                else:
                    linestyle = "-"
                    add_std_whiskers(ax, sub["size"], mean, spread, color=method_colors[m], dodge=method_dodge[m] * WHISKER_DODGE)
                # Uncomment to also show MLE's own repetition variability:
                # if m == "mle": add_std_whiskers(ax, sub["size"], mean, spread, color=method_colors[m], dodge=method_dodge["mle"] * WHISKER_DODGE)
                ax.plot(
                    sub["size"], mean, color=method_colors[m], linewidth=CURVE_LINEWIDTH, linestyle=linestyle, label=method_labels[m]
                )

            if i == 0:
                ax.set_title(col_label(p, a), fontsize=COL_TITLE_FONTSIZE)
            if j == 0:
                ax.set_ylabel(f"{row_label(nc, ess)}\n{ylabel}", fontsize=ROW_LABEL_FONTSIZE)
            if i == len(row_vals) - 1:
                ax.set_xlabel("$N$")
            if ylim:
                ax.set_ylim(*ylim)

    axes[0][-1].legend(loc="best")
    plt.tight_layout()
    return fig


## Grafico 1: pairwise F1 (network-wide, micro-averaged)

Every client pair, of every mechanism $(X, \pi_X)$, in one repetition is
pooled into a single confusion matrix before computing precision/recall/F1
(see `src/global_opt.py::pairwise_confusion`): the natural aggregation
here, since P/R/F1 are properties of individual client pairs and every
mechanism contributes the same number of pairs. Shown via sparse whisker
markers for the across-repetition spread (mean $\pm$ std or median $\pm$
Q1/Q3, depending on `CENTRAL_STAT`).


In [ ]:
# fig = plot_metric("f1", "Pairwise F1", ylim=(-0.05, 1.05))
# plt.show()
# fig.savefig(f"{res_path}_f1.svg", dpi=1200, bbox_inches="tight", transparent=False)


## Grafico 1.1: precision (network-wide, micro-averaged)

Of all the client pairs a method declared *same* (same mechanism), what
fraction really are (pooled across every mechanism, same micro-averaging as
F1; see `src/global_opt.py::pairwise_confusion`). Low precision means the
method over-merges: it fuses clients that are actually different. Same
across-repetition whisker treatment as the other network-wide plots (mean
$\pm$ std or median $\pm$ Q1/Q3, depending on `CENTRAL_STAT`).


In [ ]:
# fig = plot_metric("precision", "Precision", ylim=(-0.05, 1.05))
# plt.show()
# fig.savefig(f"{res_path}_precision.svg", dpi=1200, bbox_inches="tight", transparent=False)


## Grafico 1.2: recall (network-wide, micro-averaged)

Of all the client pairs that really *are* the same, what fraction a method
correctly found (same pooling as precision/F1). Low recall means the
method is too conservative/fragmented: it keeps apart clients that
actually share the same mechanism. Precision and recall together explain
F1's movements: a method can show a plausible-looking cluster count
(Grafico 3) while still merging or splitting the wrong clients, which
only shows up once precision and recall are examined separately.


In [ ]:
# fig = plot_metric("recall", "Recall", ylim=(-0.05, 1.05))
# plt.show()
# fig.savefig(f"{res_path}_recall.svg", dpi=1200, bbox_inches="tight", transparent=False)


## Grafico 2: Adjusted Rand Index (macro-averaged over mechanisms)

Unlike F1, ARI compares two *partitions* of the same client set, so it
cannot be pooled across mechanisms (each mechanism has its own true
partition): computed per mechanism, then macro-averaged (plain mean) across
the network's mechanisms (see `src/global_opt.py::ari_ami`), then, as
always, shown via sparse whisker markers for the across-repetition spread
(mean $\pm$ std or median $\pm$ Q1/Q3, depending on `CENTRAL_STAT`).


In [ ]:
# fig = plot_metric("ari", "ARI")
# plt.show()
# fig.savefig(f"{res_path}_ari.svg", dpi=1200, bbox_inches="tight", transparent=False)


## Grafico 2.1: Adjusted Mutual Information (macro-averaged over mechanisms)

Same aggregation as ARI (per-mechanism, then macro-averaged, since, like
ARI, it compares two *partitions* of the same client set and so cannot
be pooled across mechanisms): computed via `src/global_opt.py::ari_ami`,
then shown via sparse whisker markers for the across-repetition spread
(mean $\pm$ std or median $\pm$ Q1/Q3, depending on `CENTRAL_STAT`), same
as every other network-wide plot. Unlike ARI (a pair-counting measure),
AMI is information-theoretic
(based on mutual information between the two partitions, adjusted for
chance); the two usually agree, but AMI is less sensitive to the skewed
cluster-size distributions this task tends to produce (one large
mask-preserving cluster plus several singletons), where ARI is known to
behave less intuitively, so it is worth checking side by side with ARI
rather than assuming they always tell the same story.


In [ ]:
fig = plot_metric("ami", "AMI", ylim=(-.2, .3), hline=0.0)
plt.show()
fig.savefig(f"{res_path}_ami.svg", dpi=1200, bbox_inches="tight", transparent=False)


## Grafico 3: mean number of clusters found (diagnostic)

Not a headline accuracy metric, but a useful diagnostic for *why* F1/ARI
move the way they do: how many clusters each method finds, network-wide,
versus the true mean cluster count (dashed). `mle`'s baseline is handed the
true cluster count as an oracle (`cluster_1d_wcss_optimal`, see
`src/global_opt.py`), deliberately conservative, i.e. an advantage MOSAIC
does NOT get, so its curve sits exactly on top of "True" by construction;
it is shown mainly as a visual anchor for the other three methods, which
get no such help.


In [ ]:
fig, axes = plt.subplots(
    len(row_vals),
    len(col_vals),
    figsize=(SUBPLOT_WIDTH * len(col_vals), SUBPLOT_HEIGHT * len(row_vals)),
    squeeze=False,
    sharex=True,
    sharey=True,
)
for i, (nc, ess) in enumerate(row_vals):
    for j, (p, a) in enumerate(col_vals):
        ax = axes[i][j]
        sub = df_mean[
            (df_mean["prob_shift"] == p)
            & (df_mean["alpha"] == a)
            & (df_mean["n_clients"] == nc)
            & (df_mean["ess"] == ess)
        ].sort_values("size")

        # True uses MLE's dodge slot (same canonical order as every other
        # plot): the two never coexist here, MLE itself is never drawn
        # (see Grafico 3's own markdown for why).
        true_mean, true_spread = center_spread(sub, "n_clusters_true_mean")
        ax.plot(sub["size"], true_mean, linestyle="--", linewidth=CURVE_LINEWIDTH, color="grey", label="True")
        add_std_whiskers(ax, sub["size"], true_mean, true_spread, color="grey", dodge=method_dodge["mle"] * WHISKER_DODGE)

        for m in METHODS:
            if m == "mle":
                continue
            mean, spread = center_spread(sub, f"{m}_n_clusters_mean")
            ax.plot(
                sub["size"],
                mean,
                color=method_colors[m],
                label=method_labels[m],
                linewidth=CURVE_LINEWIDTH,
                linestyle="-"
            )
            add_std_whiskers(ax, sub["size"], mean, spread, color=method_colors[m], dodge=method_dodge[m] * WHISKER_DODGE)

        if i == 0:
            ax.set_title(col_label(p, a), fontsize=COL_TITLE_FONTSIZE)
        if j == 0:
            ax.set_ylabel(f"{row_label(nc, ess)}\nMean cluster count", fontsize=ROW_LABEL_FONTSIZE)
        if i == len(row_vals) - 1:
            ax.set_xlabel("$N$")

axes[0][-1].legend()
plt.tight_layout()
plt.show()
fig.savefig(f"{res_path}_nclusters.svg", dpi=1200, bbox_inches="tight", transparent=False)
